# LocalTransform 模型架构与推理展示

**LocalTransform** 是一种基于**局部反应模板 + GNN**的正向反应产物预测模型（Nature Machine Intelligence 2022）。与传统全局模板方法不同，它在**每条化学键**上独立预测局部模板，再组合为完整的反应预测。

```text
反应物 SMILES
    │
    ▼
┌───────────────────┐
│  DGL 分子图构建     │   ← WeaveAtomFeaturizer + CanonicalBondFeaturizer
│  + ADM 距离矩阵     │   ← Atom Distance Matrix (截断到 6 跳)
│  + 虚拟/真实键列表   │
└─────────┬─────────┘
          │
          ▼
┌───────────────────┐
│  MPNN 消息传递      │   ← 3 步消息传递 (DGL-Life MPNNGNN)
│  (节点嵌入 256d)    │
└─────────┬─────────┘
          │
          ▼
┌───────────────────┐
│  Global Reactivity │   ← 3 层 × 8 头注意力 + ADM 位置编码
│  Attention (原子级) │
└─────────┬─────────┘
          │
          ▼
┌───────────────────┐
│  Reactive Pooling  │   ← 为每条键评分，选择 Top-K 活性键
│  (虚拟键 + 真实键)  │
└────┬──────────┬───┘
     │          │
     ▼          ▼
 pooling_v   pooling_r    ← 二分类头：该键是否为反应位点
 bondnet_v   bondnet_r    ← 提取键级特征
     │          │
     ▼          ▼
┌───────────────────┐
│  Global Reactivity │   ← 3 层 × 8 头注意力 + BDM 位置编码
│  Attention (键级)  │
└────┬──────────┬───┘
     │          │
     ▼          ▼
  output_v   output_r    ← 多分类头: 预测局部模板 ID
  (2536类)   (4371类)     (含 class 0 = 无反应)
     │          │
     ▼          ▼
  Softmax → combined_edit → 模板应用 → 产物 SMILES
```

## 本 Notebook 内容

| 章节 | 内容 | 源码位置 |
|------|------|---------|
| Step 1 | 环境与导入 | — |
| Step 2 | MPNN 消息传递网络 | `dgllife.model.MPNNGNN` |
| Step 3 | 多头注意力与相对位置编码 | `scripts/model_utils.py → MultiHeadAttention` |
| Step 4 | 全局反应性注意力 (GRA) | `scripts/model_utils.py → Global_Reactivity_Attention` |
| Step 5 | 反应性池化 (Reactive Pooling) | `scripts/model_utils.py → reactive_pooling` |
| Step 6 | 完整 LocalTransform 模型 | `scripts/models.py → LocalTransform` |
| Step 7 | 加载预训练模型进行推理 | `Synthesis.py → localtransform` |

**参考论文**: Shuan Chen & Yousung Jung, *"A Generalized Template-Based Graph Neural Network for Accurate Organic Reactivity Prediction"*, Nature Machine Intelligence, 2022

In [1]:
# ============================================================
# Step 1: 环境与导入
# ============================================================
import os, sys, math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn


def find_project_root(start=None):
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / 'teaching_demos').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')


PROJECT_ROOT = find_project_root()
LT_SOURCE = PROJECT_ROOT / 'source_repos' / 'LocalTransform'
LT_ENV = PROJECT_ROOT / 'envs' / 'localtransform_tutorial_envs'
DATA_DIR = LT_SOURCE / 'data' / 'USPTO_480k'

for path in [LT_SOURCE, LT_SOURCE / 'scripts']:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

# 设备（教学仅 CPU）
device = 'cpu'

print(f"PyTorch 版本: {torch.__version__}")
print(f"设备: {device}")
print(f"当前 Python: {sys.executable}")
print(f"LocalTransform 源码: {LT_SOURCE}")

# DGL
import dgl
from dgllife.model import MPNNGNN
from dgllife.utils import WeaveAtomFeaturizer, CanonicalBondFeaturizer, mol_to_bigraph
print(f"DGL 版本: {dgl.__version__}")

# RDKit
from rdkit import Chem
print(f"RDKit 版本: {Chem.rdBase.rdkitVersion}")

expected_python = LT_ENV / 'bin' / 'python'
if expected_python.exists() and Path(sys.executable).resolve() != expected_python.resolve():
    print('警告: 当前 kernel 不是 localtransform_tutorial_envs，若后续导入失败请切换内核。')

print("\n✅ 环境就绪")

PyTorch 版本: 2.5.1
设备: cpu
当前 Python: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/localtransform_tutorial_envs/bin/python
LocalTransform 源码: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/LocalTransform


/home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/localtransform_tutorial_envs/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DGL 版本: 2.4.0+cu121
RDKit 版本: 2025.09.6

✅ 环境就绪


## Step 2: MPNN 消息传递网络

LocalTransform 的第一阶段使用 **MPNNGNN**（来自 DGL-Life），对分子图中的每个原子节点进行 $T$ 步消息传递，生成包含局部化学环境信息的节点嵌入。

### 消息传递公式

对于每一步 $t = 1, \dots, T$：

$$
m_v^{(t)} = \sum_{u \in \mathcal{N}(v)} M_t(h_v^{(t-1)}, h_u^{(t-1)}, e_{vu})
$$

$$
h_v^{(t)} = U_t(h_v^{(t-1)}, m_v^{(t)})
$$

其中：

- $h_v^{(0)}$ 是 `WeaveAtomFeaturizer` 生成的原子特征（约 42 维）
- $e_{vu}$ 是 `CanonicalBondFeaturizer` 生成的键特征（约 12 维，含 self-loop）
- $M_t$ 和 $U_t$ 分别是消息函数和更新函数（均为神经网络）
- $T = 3$（默认 `num_step_message_passing=3`）

### 默认超参数

| 参数 | 值 | 说明 |
|------|------|------|
| `node_out_feats` | 256 | 输出节点特征维度 |
| `edge_hidden_feats` | 32 | 边网络隐藏层维度 |
| `num_step_message_passing` | 3 | 消息传递步数 |

> **源码位置**: `scripts/models.py` 第 27-31 行，`self.mpnn = MPNNGNN(...)`

In [2]:
# ============================================================
# Step 2: MPNN 消息传递网络演示
# ============================================================
from functools import partial

# 构建特征化器（与源码 utils.py init_featurizer 一致）
atom_types = ['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe',
    'As', 'Al', 'I', 'B', 'V', 'K', 'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti',
    'Zn', 'H', 'Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd', 'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb',
    'W', 'Ru', 'Nb', 'Re', 'Te', 'Rh', 'Ta', 'Tc', 'Ba', 'Bi', 'Hf', 'Mo', 'U', 'Sm', 'Os', 'Ir',
    'Ce', 'Gd', 'Ga', 'Cs']

node_featurizer = WeaveAtomFeaturizer(atom_types=atom_types)
edge_featurizer = CanonicalBondFeaturizer(self_loop=True)

# 以苯甲醛为例
smiles = 'O=Cc1ccccc1'
mol = Chem.MolFromSmiles(smiles)
graph = mol_to_bigraph(mol, add_self_loop=True,
                       node_featurizer=node_featurizer,
                       edge_featurizer=edge_featurizer,
                       canonical_atom_order=False)

node_feats = graph.ndata['h']
edge_feats = graph.edata['e']

print(f"分子: {smiles}")
print(f"原子数: {graph.num_nodes()}, 边数 (含 self-loop): {graph.num_edges()}")
print(f"节点特征维度: {node_feats.shape[-1]}")
print(f"边特征维度:   {edge_feats.shape[-1]}")

# 构建 MPNNGNN（与 default_config.json 一致）
node_out_feats = 256
edge_hidden_feats = 32
num_step_message_passing = 3

mpnn = MPNNGNN(
    node_in_feats=node_feats.shape[-1],
    node_out_feats=node_out_feats,
    edge_in_feats=edge_feats.shape[-1],
    edge_hidden_feats=edge_hidden_feats,
    num_step_message_passing=num_step_message_passing
)

# 前向传播
with torch.no_grad():
    atom_feats = mpnn(graph, node_feats, edge_feats)

print(f"\nMPNN 输出维度: {atom_feats.shape}")
print(f"  → 每个原子获得一个 {node_out_feats} 维的嵌入向量")
print(f"\nMPNN 参数量: {sum(p.numel() for p in mpnn.parameters()):,}")

# 打印每个原子的嵌入统计
for i, atom in enumerate(mol.GetAtoms()):
    feat = atom_feats[i]
    print(f"  原子 {i} ({atom.GetSymbol()}): mean={feat.mean():.4f}, std={feat.std():.4f}, norm={feat.norm():.4f}")

print("\n✅ MPNN 验证完成")

分子: O=Cc1ccccc1
原子数: 8, 边数 (含 self-loop): 24
节点特征维度: 80
边特征维度:   13

MPNN 输出维度: torch.Size([8, 256])
  → 每个原子获得一个 256 维的嵌入向量

MPNN 参数量: 2,578,880
  原子 0 (O): mean=-0.0084, std=0.1415, norm=2.2642
  原子 1 (C): mean=-0.0089, std=0.2275, norm=3.6356
  原子 2 (C): mean=-0.0137, std=0.3882, norm=6.2027
  原子 3 (C): mean=-0.0081, std=0.3631, norm=5.7995
  原子 4 (C): mean=-0.0065, std=0.3516, norm=5.6161
  原子 5 (C): mean=-0.0053, std=0.3503, norm=5.5939
  原子 6 (C): mean=-0.0065, std=0.3516, norm=5.6161
  原子 7 (C): mean=-0.0081, std=0.3631, norm=5.7995

✅ MPNN 验证完成


## Step 3: 多头注意力与相对位置编码 (MultiHeadAttention)

这是 LocalTransform 的核心创新之一。标准 Transformer 使用绝对位置编码，而 **LocalTransform 使用基于图距离的相对位置编码**。

### 注意力公式

$$
\text{Attn}(Q, K, V) = \text{softmax}\left(\frac{QK^T + QR^T}{\sqrt{d_k}}\right) V
$$

其中，$R$ 是**相对位置嵌入矩阵**，由 Atom Distance Matrix (ADM) 索引：

- ADM 中 `distance[i][j] = k` 表示原子 `i` 和原子 `j` 之间的最短路径为 `k` 跳
- 截断到 `max_distance=6`，跨分子为 `7`
- 对应 `positional_number=8`（`0-7` 共 8 个位置类）

### 门控机制

输出还通过 **Gating** 机制进行调制：

$$
\text{output} = W_o \cdot (\text{Attn} \odot \sigma(W_g \cdot x))
$$

其中，$\sigma$ 是 Sigmoid 函数。这允许模型选择性地保留或抑制注意力信息。

> **源码位置**: `scripts/model_utils.py`，`class MultiHeadAttention`

In [4]:
# ============================================================
# Step 3: 多头注意力与相对位置编码演示
# ============================================================
from scripts.model_utils import MultiHeadAttention

# 创建一个注意力层（参数与源码一致）
d_model = 256          # node_out_feats
heads = 8              # attention_heads
positional_number = 8  # 对应 ADM 的 0~7 距离

attn_layer = MultiHeadAttention(
    heads=heads,
    d_model=d_model,
    positional_number=positional_number,
    dropout=0.1
)

print("MultiHeadAttention 结构:")
print(f"  注意力头数: {heads}")
print(f"  每头维度 d_k: {d_model // heads} = {d_model}/{heads}")
print(f"  相对位置类数: {positional_number} (ADM 距离 0~7)")
print(f"  参数量: {sum(p.numel() for p in attn_layer.parameters()):,}")

# 构建演示输入
# 假设 batch=1, 有 8 个原子, 节点特征 256 维
n_atoms = 8
x = torch.randn(1, n_atoms, d_model)

# 构建 ADM (原子距离矩阵)
# 模拟一条链式分子: 距离 = |i - j|，截断到 7
adm = torch.zeros(1, n_atoms, n_atoms, dtype=torch.long)
for i in range(n_atoms):
    for j in range(n_atoms):
        adm[0, i, j] = min(abs(i - j), 7)

# 掩码: 所有原子都有效
mask = torch.ones(1, n_atoms, dtype=torch.bool)

print(f"\n输入形状: x={x.shape}, ADM={adm.shape}, mask={mask.shape}")
print(f"\nADM (8个原子的距离矩阵):")
print(adm[0].numpy())

# 前向传播
# MultiHeadAttention.forward() 期望 gpm 形状为 (batch, n_atoms, n_atoms)
attn_layer.eval()
with torch.no_grad():
    output, att_scores = attn_layer(x, adm, mask)

print(f"\n输出形状: {output.shape}")
print(f"注意力权重形状: {att_scores.shape}")
print(f"  → (batch={att_scores.shape[0]}, heads={att_scores.shape[1]}, atoms={att_scores.shape[2]}, atoms={att_scores.shape[3]})")

# 可视化第 0 个头的注意力权重
print(f"\n注意力权重 (Head 0, 前4个原子):")
att_head0 = att_scores[0, 0, :4, :].numpy()
print(np.array2string(att_head0, precision=3, suppress_small=True))

# 验证每行和为 1
row_sums = att_scores[0, 0].sum(dim=-1).numpy()
print(f"\n每行注意力权重之和 (Head 0): {row_sums.round(4)}")
print("  → 应该全部接近 1.0")

# 展示相对位置嵌入
print(f"\n相对位置嵌入 relative_k 形状: {attn_layer.relative_k.shape}")
print(f"  → ({positional_number} 个距离类, {d_model // heads} 维)")

print("\n✅ MultiHeadAttention 验证完成")

MultiHeadAttention 结构:
  注意力头数: 8
  每头维度 d_k: 32 = 256/8
  相对位置类数: 8 (ADM 距离 0~7)
  参数量: 395,008

输入形状: x=torch.Size([1, 8, 256]), ADM=torch.Size([1, 8, 8]), mask=torch.Size([1, 8])

ADM (8个原子的距离矩阵):
[[0 1 2 3 4 5 6 7]
 [1 0 1 2 3 4 5 6]
 [2 1 0 1 2 3 4 5]
 [3 2 1 0 1 2 3 4]
 [4 3 2 1 0 1 2 3]
 [5 4 3 2 1 0 1 2]
 [6 5 4 3 2 1 0 1]
 [7 6 5 4 3 2 1 0]]

输出形状: torch.Size([1, 8, 256])
注意力权重形状: torch.Size([1, 8, 8, 8])
  → (batch=1, heads=8, atoms=8, atoms=8)

注意力权重 (Head 0, 前4个原子):
[[0.551 0.108 0.058 0.017 0.007 0.091 0.043 0.124]
 [0.743 0.023 0.002 0.005 0.075 0.112 0.027 0.014]
 [0.369 0.05  0.034 0.01  0.042 0.133 0.217 0.144]
 [0.059 0.088 0.319 0.167 0.124 0.065 0.061 0.117]]

每行注意力权重之和 (Head 0): [1. 1. 1. 1. 1. 1. 1. 1.]
  → 应该全部接近 1.0

相对位置嵌入 relative_k 形状: torch.Size([8, 32])
  → (8 个距离类, 32 维)

✅ MultiHeadAttention 验证完成


## Step 4: 全局反应性注意力 (Global Reactivity Attention)

Global Reactivity Attention (GRA) 是将多个注意力层堆叠起来的模块。LocalTransform 中使用**两次 GRA**：

| GRA 实例 | 输入 | 位置编码 | 作用 |
|----------|------|---------|------|
| `atom_att` | 原子嵌入 | ADM (`positional=8`) | 原子级全局信息聚合 |
| `bond_att` | 键级特征 | BDM (`positional=2`) | 键级全局信息聚合 |

### 每一层的计算流程

```text
                    x
                    │
              LayerNorm
                    │
           ┌────────┴────────┐
           │  MultiHead      │
           │  Attention      │──► m (注意力输出)
           │  (含 ADM/BDM    │
           │   位置编码)     │
           └─────────────────┘
                    │
                x + m
                    │
              LayerNorm
                    │
           ┌────────┴────────┐
           │  FeedForward    │
           │  (d → 2d → d)   │──► x' = x + FFN(x + m)
           │  + GELU + Drop  │
           └─────────────────┘
                    │
                  x' (下一层输入)
```

默认堆叠 **3 层**（`attention_layers=3`）。

### BDM (Bond Distance Matrix)

键级注意力中的位置编码使用 **BDM**，而非 ADM：

- `BDM[i][j] = 1`，如果两条键共享至少一个原子
- `BDM[i][j] = 0`，否则
- `positional_number = 2`（仅 `0/1` 两类）

> **源码位置**: `scripts/model_utils.py`，`class Global_Reactivity_Attention`、`class FeedForward`、`class GELU`

In [6]:
# ============================================================
# Step 4: 全局反应性注意力 (GRA) 演示
# ============================================================
from scripts.model_utils import Global_Reactivity_Attention, FeedForward, GELU

# 构建 GRA 模块（原子级，与源码 models.py 第 33 行一致）
atom_gra = Global_Reactivity_Attention(
    d_model=256,
    heads=8,
    n_layers=3,
    positional_number=8,  # ADM: 0~7
    dropout=0.1
)

print("原子级 Global Reactivity Attention 结构:")
print(f"  层数: {atom_gra.n_layers}")
print(f"  每层: MultiHeadAttention + FeedForward")
print(f"  位置编码类数: 8 (ADM)")
print(f"  参数量: {sum(p.numel() for p in atom_gra.parameters()):,}")

# 构建 GRA 模块（键级，与源码 models.py 第 34 行一致）
bond_gra = Global_Reactivity_Attention(
    d_model=256,
    heads=8,
    n_layers=3,
    positional_number=2,  # BDM: 0 或 1
    dropout=0.1
)

print(f"\n键级 Global Reactivity Attention:")
print(f"  位置编码类数: 2 (BDM: 是否共享原子)")
print(f"  参数量: {sum(p.numel() for p in bond_gra.parameters()):,}")

# 演示原子级 GRA 前向传播
n_atoms = 8
x = torch.randn(1, n_atoms, 256)  # MPNN 输出
adm = torch.zeros(1, n_atoms, n_atoms, dtype=torch.long)
for i in range(n_atoms):
    for j in range(n_atoms):
        adm[0, i, j] = min(abs(i - j), 7)
mask = torch.ones(1, n_atoms, dtype=torch.bool)

atom_gra.eval()
with torch.no_grad():
    out, att_scores = atom_gra(x, adm, mask)

print(f"\n原子级 GRA 前向传播:")
print(f"  输入: {x.shape}")
print(f"  输出: {out.shape}")
print(f"  注意力权重: {len(att_scores)} 层")
for layer_idx, scores in att_scores.items():
    print(f"    Layer {layer_idx}: {scores.shape}")

# 演示 FeedForward 和 GELU
print(f"\nFeedForward 结构 (d=256):")
ff = FeedForward(256)
print(f"  Linear(256 → 512) → GELU → Dropout → Linear(512 → 256)")
print(f"  参数量: {sum(p.numel() for p in ff.parameters()):,}")

# GELU 激活函数
gelu = GELU()
x_demo = torch.linspace(-3, 3, 7)
print(f"\nGELU 激活值: {[f'{v:.3f}' for v in gelu(x_demo).tolist()]}")

print("\n✅ GRA 验证完成")

原子级 Global Reactivity Attention 结构:
  层数: 3
  每层: MultiHeadAttention + FeedForward
  位置编码类数: 8 (ADM)
  参数量: 1,975,296

键级 Global Reactivity Attention:
  位置编码类数: 2 (BDM: 是否共享原子)
  参数量: 1,974,720

原子级 GRA 前向传播:
  输入: torch.Size([1, 8, 256])
  输出: torch.Size([1, 8, 256])
  注意力权重: 3 层
    Layer 0: torch.Size([1, 8, 8, 8])
    Layer 1: torch.Size([1, 8, 8, 8])
    Layer 2: torch.Size([1, 8, 8, 8])

FeedForward 结构 (d=256):
  Linear(256 → 512) → GELU → Dropout → Linear(512 → 256)
  参数量: 263,424

GELU 激活值: ['-0.004', '-0.045', '-0.159', '0.000', '0.841', '1.955', '2.996']

✅ GRA 验证完成


## Step 5: 反应性池化 (Reactive Pooling)

这是 LocalTransform 的关键机制：**从所有候选键中筛选出最可能的反应活性键**。

### 虚拟键 vs 真实键

| 类型 | 定义 | 反应类型 | 模板数量 |
|------|------|---------|---------|
| 虚拟键 (Virtual) | 不相连的原子对 `(a, b)` | 键形成 (A = Addition) | 2535 |
| 真实键 (Real) | 已存在的化学键 `(a, b)` | 键断裂 (B)、键变化 (C)、键消除 (R) | 4370 |

### 池化流程

```text
                     所有虚拟键对                所有真实键对
                         │                         │
                    ┌────┴────┐               ┌────┴────┐
                    │ concat  │               │ concat  │
                    │ (h_a,h_b)│              │ (h_a,h_b)│
                    └────┬────┘               └────┬────┘
                         │                         │
                         ▼                         ▼
                    pooling_v                  pooling_r
                 (512→256→2)              (512→256→2)
                    │                          │
               二分类: 是否反应?           二分类: 是否反应?
                    │                          │
                 Top-K                      Top-K
                    │                          │
                    ▼                          ▼
                 bondnet_v                 bondnet_r
              (512→256→256)            (512→256→256)
                    │                          │
                    └──────────┬───────────────┘
                               │
                          拼接后送入键级 GRA
```

- `pool_size = n_atoms`（每个分子最多选择 `n_atoms` 条最活跃的键）
- `pooling_v` / `pooling_r` 输出 2 维：`[非活性分数, 活性分数]`
- 选择“活性分数”最高的 Top-K 条键

> **源码位置**: `scripts/model_utils.py`，`reactive_pooling()`、`pack_bond_feats()`、`get_bdm()`
>
> **模型中**: `scripts/models.py`，`self.pooling_v`、`self.pooling_r`、`self.bondnet_v`、`self.bondnet_r`

In [8]:
# ============================================================
# Step 5: 反应性池化演示
# ============================================================
from scripts.dataset import get_bonds, get_adm
from scripts.model_utils import pack_atom_feats, reactive_pooling, pack_bond_feats, get_bdm

# 以乙酸乙酯为例
smiles = 'CCOC(=O)C'
mol = Chem.MolFromSmiles(smiles)
n_atoms = mol.GetNumAtoms()
print(f"分子: {smiles}")
print(f"原子数: {n_atoms}")

# 获取虚拟键和真实键
v_bonds, r_bonds = get_bonds(smiles)
print(f"\n虚拟键 (不相连原子对): {len(v_bonds)} 条")
print(f"  前 5 条: {v_bonds[:5].tolist()}")
print(f"真实键 (已有化学键):   {len(r_bonds)} 条")
print(f"  前 5 条: {r_bonds[:5].tolist()}")

# 获取 ADM
adm = get_adm(mol)
print(f"\nADM 形状: {adm.shape}")
print(f"ADM 矩阵:")
print(adm.astype(int))

# 构建 DGL 图并获取 MPNN 嵌入
graph = mol_to_bigraph(mol, add_self_loop=True,
                       node_featurizer=node_featurizer,
                       edge_featurizer=edge_featurizer,
                       canonical_atom_order=False)
with torch.no_grad():
    atom_feats = mpnn(graph, graph.ndata['h'], graph.edata['e'])
print(f"\nMPNN 输出: {atom_feats.shape}")

# pack_atom_feats: 将不等长图特征填充为 batch 矩阵
padded_feats, mask = pack_atom_feats(graph, atom_feats)
print(f"pack_atom_feats 输出: padded_feats={padded_feats.shape}, mask={mask.shape}")

# 模拟 reactive_pooling
# 需要 pooling 和 bondnet 网络
activation = GELU()
pooling_v = nn.Sequential(
    nn.Linear(256*2, 256), activation, nn.Dropout(0.2), nn.Linear(256, 2))
pooling_r = nn.Sequential(
    nn.Linear(256*2, 256), activation, nn.Dropout(0.2), nn.Linear(256, 2))
bondnet_v = nn.Sequential(
    nn.Linear(256*2, 256), activation, nn.Linear(256, 256))
bondnet_r = nn.Sequential(
    nn.Linear(256*2, 256), activation, nn.Linear(256, 256))
poolings = {'virtual': pooling_v, 'real': pooling_r}
bondnets = {'virtual': bondnet_v, 'real': bondnet_r}

# 准备 bonds_dict
bonds_dict = {
    'virtual': [torch.from_numpy(v_bonds).long()],
    'real':    [torch.from_numpy(r_bonds).long()]
}

with torch.no_grad():
    # 先经过 atom_gra（简化：直接使用 MPNN 输出）
    # 正式模型中 atom_feats 会经过 GRA 处理
    atom_gra.eval()
    adm_batch = torch.tensor(adm).unsqueeze(0).long()
    padded_feats_att, _ = atom_gra(padded_feats, adm_batch, mask)
    
    # reactive_pooling
    idxs_dict, rout_dict, bonds_feats, pooled_bonds = reactive_pooling(
        graph, padded_feats_att, bonds_dict, poolings, bondnets
    )

print(f"\n=== Reactive Pooling 结果 ===")
print(f"选中的虚拟键索引: {idxs_dict['virtual'][0].tolist()[:10]}... (共 {len(idxs_dict['virtual'][0])} 条)")
print(f"选中的真实键索引: {idxs_dict['real'][0].tolist()[:10]}... (共 {len(idxs_dict['real'][0])} 条)")

if rout_dict['virtual'].numel() > 0:
    print(f"\n虚拟键活性分数 (前 5 条): {torch.softmax(rout_dict['virtual'][:5], dim=-1)[:, 1].tolist()}")
if rout_dict['real'].numel() > 0:
    print(f"真实键活性分数 (前 5 条): {torch.softmax(rout_dict['real'][:5], dim=-1)[:, 1].tolist()}")

print(f"\n池化后键特征数量: {len(bonds_feats[0])}")
print(f"池化后键特征维度: {bonds_feats[0].shape[-1]}")

# 展示 BDM
print(f"\n=== Bond Distance Matrix (BDM) ===")
bdm = get_bdm(pooled_bonds[0], len(bonds_feats[0]))
print(f"BDM 形状: {bdm.shape}")
print(f"BDM (前 6x6):")
print(bdm[0, :6, :6].numpy())
print("  → 1 = 两条键共享至少一个原子, 0 = 不共享")

# pack_bond_feats
padded_bond_feats, bond_mask, bcms = pack_bond_feats(bonds_feats, pooled_bonds)
print(f"\npack_bond_feats 输出:")
print(f"  padded_bond_feats: {padded_bond_feats.shape}")
print(f"  bond_mask: {bond_mask.shape}")
print(f"  bcms (BDM batch): {bcms.shape}")

print("\n✅ Reactive Pooling 验证完成")

分子: CCOC(=O)C
原子数: 6

虚拟键 (不相连原子对): 20 条
  前 5 条: [[0, 2], [0, 3], [0, 4], [0, 5], [1, 3]]
真实键 (已有化学键):   10 条
  前 5 条: [[0, 1], [1, 0], [1, 2], [2, 1], [2, 3]]

ADM 形状: (6, 6)
ADM 矩阵:
[[0 1 2 3 4 4]
 [1 0 1 2 3 3]
 [2 1 0 1 2 2]
 [3 2 1 0 1 1]
 [4 3 2 1 0 2]
 [4 3 2 1 2 0]]

MPNN 输出: torch.Size([6, 256])
pack_atom_feats 输出: padded_feats=torch.Size([1, 6, 256]), mask=torch.Size([1, 6])

=== Reactive Pooling 结果 ===
选中的虚拟键索引: [6, 8, 12, 7, 1, 15]... (共 6 条)
选中的真实键索引: [7, 8, 1, 2, 0, 6]... (共 6 条)

虚拟键活性分数 (前 5 条): [0.5423585176467896, 0.545799970626831, 0.5501565337181091, 0.5222545862197876, 0.5357934236526489]
真实键活性分数 (前 5 条): [0.49410879611968994, 0.48846185207366943, 0.4872835874557495, 0.48589640855789185, 0.48452597856521606]

池化后键特征数量: 12
池化后键特征维度: 256

=== Bond Distance Matrix (BDM) ===
BDM 形状: torch.Size([1, 12, 12])
BDM (前 6x6):
[[1 0 0 0 0 1]
 [0 1 1 1 0 1]
 [0 1 1 1 1 1]
 [0 1 1 1 1 0]
 [0 0 1 1 1 0]
 [1 1 1 0 0 1]]
  → 1 = 两条键共享至少一个原子, 0 = 不共享

pack_bond_feats 输出:
  padded_b

## Step 6: 完整 LocalTransform 模型

将上述组件组装为完整的 `LocalTransform` 模型。

### 完整前向传播流程

```python
def forward(bg, adms, bonds_dict, node_feats, edge_feats):
    # 1. MPNN 消息传递: 原子特征 → 256d 嵌入
    atom_feats = self.mpnn(bg, node_feats, edge_feats)

    # 2. Pack: 不等长图 → 填充批量矩阵
    atom_feats, mask = pack_atom_feats(bg, atom_feats)

    # 3. 原子级 GRA: 全局信息聚合 (3 层 × 8 头, ADM 位置编码)
    atom_feats, atom_attention = self.atom_att(atom_feats, adms, mask)

    # 4. Reactive Pooling: 选择 Top-K 活性键 + 提取键特征
    idxs_dict, rout_dict, bonds_feats, bonds = reactive_pooling(...)

    # 5. Pack bonds: 键特征 → 填充批量矩阵 + BDM
    bond_feats, mask, bcms = pack_bond_feats(bonds_feats, bonds)

    # 6. 键级 GRA: 键间信息聚合 (3 层 × 8 头, BDM 位置编码)
    bond_feats, bond_attention = self.bond_att(bond_feats, bcms, mask)

    # 7. Unpack: 分离虚拟键和真实键特征
    feats_v, feats_r = unpack_bond_feats(bond_feats, idxs_dict)

    # 8. 模板分类: 多分类 softmax
    template_v = self.output_v(feats_v)  # → (n_v_bonds, 2536)
    template_r = self.output_r(feats_r)  # → (n_r_bonds, 4371)

    return template_v, template_r, ...
```

### 输出说明

模型返回 **7 个值**：

| 输出 | 形状 | 说明 |
|------|------|------|
| `template_v` | `(n_v, Template_vn+1)` | 虚拟键模板分类 logits |
| `template_r` | `(n_r, Template_rn+1)` | 真实键模板分类 logits |
| `rout_v` | `(n_v, 2)` | 虚拟键活性分数 |
| `rout_r` | `(n_r, 2)` | 真实键活性分数 |
| `idxs_v` | `list` | 虚拟键 Top-K 索引 |
| `idxs_r` | `list` | 真实键 Top-K 索引 |
| `attentions` | `tuple` | `(原子注意力, 键注意力)` |

> **源码位置**: `scripts/models.py`，`class LocalTransform`

In [9]:
# ============================================================
# Step 6: 完整 LocalTransform 模型演示
# ============================================================
import pandas as pd

# 读取模板统计
real_templates = pd.read_csv(os.path.join(DATA_DIR, 'real_templates.csv'))
virtual_templates = pd.read_csv(os.path.join(DATA_DIR, 'virtual_templates.csv'))
Template_rn = len(real_templates)
Template_vn = len(virtual_templates)
print(f"真实键模板数: {Template_rn}")
print(f"虚拟键模板数: {Template_vn}")

# 读取配置
import json
with open(os.path.join(LT_SOURCE, 'data', 'configs', 'default_config.json')) as f:
    config = json.load(f)
print(f"\n默认配置: {json.dumps(config, indent=2)}")

# 构建完整模型
from scripts.models import LocalTransform

model = LocalTransform(
    node_in_feats=node_featurizer.feat_size(),
    edge_in_feats=edge_featurizer.feat_size(),
    node_out_feats=config['node_out_feats'],           # 256
    edge_hidden_feats=config['edge_hidden_feats'],     # 32
    num_step_message_passing=config['num_step_message_passing'],  # 3
    attention_heads=config['attention_heads'],          # 8
    attention_layers=config['attention_layers'],        # 3
    Template_rn=Template_rn,                           # 4370
    Template_vn=Template_vn                            # 2535
)
model = model.to(device)
model.eval()

# 打印模型结构
total_params = sum(p.numel() for p in model.parameters())
print(f"\n=== LocalTransform 模型参数统计 ===")
print(f"总参数量: {total_params:,}")
for name, module in [
    ('MPNN', model.mpnn),
    ('原子级 GRA', model.atom_att),
    ('键级 GRA', model.bond_att),
    ('Pooling_v (虚拟键活性)', model.pooling_v),
    ('Pooling_r (真实键活性)', model.pooling_r),
    ('BondNet_v (虚拟键特征)', model.bondnet_v),
    ('BondNet_r (真实键特征)', model.bondnet_r),
    ('Output_v (虚拟模板分类)', model.output_v),
    ('Output_r (真实模板分类)', model.output_r),
]:
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {n_params:,} ({n_params/total_params*100:.1f}%)")

# 前向传播演示
smiles = 'CCOC(=O)C'
mol = Chem.MolFromSmiles(smiles)
graph = mol_to_bigraph(mol, add_self_loop=True,
                       node_featurizer=node_featurizer,
                       edge_featurizer=edge_featurizer,
                       canonical_atom_order=False)
bg = dgl.batch([graph])
adms = torch.tensor(get_adm(mol)).unsqueeze(0).long()
v_bonds, r_bonds = get_bonds(smiles)
bonds_dicts = {
    'virtual': [torch.from_numpy(v_bonds).long()],
    'real':    [torch.from_numpy(r_bonds).long()]
}
node_f = bg.ndata.pop('h')
edge_f = bg.edata.pop('e')

with torch.no_grad():
    template_v, template_r, rout_v, rout_r, idxs_v, idxs_r, attentions = \
        model(bg, adms, bonds_dicts, node_f, edge_f)

print(f"\n=== 前向传播结果 ({smiles}) ===")
print(f"虚拟键模板 logits: {template_v.shape}")
print(f"真实键模板 logits: {template_r.shape}")
print(f"虚拟键活性分数: {rout_v.shape}")
print(f"真实键活性分数: {rout_r.shape}")
print(f"选中虚拟键数: {len(idxs_v[0])}")
print(f"选中真实键数: {len(idxs_r[0])}")
print(f"注意力层数: 原子级 {len(attentions[0])} 层, 键级 {len(attentions[1])} 层")

# 展示预测结果
v_probs = torch.softmax(template_v, dim=-1)
r_probs = torch.softmax(template_r, dim=-1)
print(f"\n虚拟键 Top-3 模板预测:")
for i in range(min(3, template_v.shape[0])):
    top_vals, top_ids = v_probs[i].topk(3)
    bond_idx = idxs_v[0][i].item()
    bond_pair = v_bonds[bond_idx]
    print(f"  键 ({bond_pair[0]},{bond_pair[1]}): {[(id.item(), f'{val:.4f}') for id, val in zip(top_ids, top_vals)]}")

print(f"\n真实键 Top-3 模板预测:")
for i in range(min(3, template_r.shape[0])):
    top_vals, top_ids = r_probs[i].topk(3)
    bond_idx = idxs_r[0][i].item()
    bond_pair = r_bonds[bond_idx]
    print(f"  键 ({bond_pair[0]},{bond_pair[1]}): {[(id.item(), f'{val:.4f}') for id, val in zip(top_ids, top_vals)]}")
print("  → class 0 = 无反应")

print("\n✅ 完整模型验证完成")

真实键模板数: 4370
虚拟键模板数: 2535

默认配置: {
  "attention_heads": 8,
  "attention_layers": 3,
  "edge_hidden_feats": 32,
  "node_out_feats": 256,
  "num_step_message_passing": 3
}

=== LocalTransform 模型参数统计 ===
总参数量: 9,093,503
  MPNN: 2,578,880 (28.4%)
  原子级 GRA: 1,975,296 (21.7%)
  键级 GRA: 1,974,720 (21.7%)
  Pooling_v (虚拟键活性): 131,842 (1.4%)
  Pooling_r (真实键活性): 131,842 (1.4%)
  BondNet_v (虚拟键特征): 197,120 (2.2%)
  BondNet_r (真实键特征): 197,120 (2.2%)
  Output_v (虚拟模板分类): 717,544 (7.9%)
  Output_r (真实模板分类): 1,189,139 (13.1%)

=== 前向传播结果 (CCOC(=O)C) ===
虚拟键模板 logits: torch.Size([6, 2536])
真实键模板 logits: torch.Size([6, 4371])
虚拟键活性分数: torch.Size([6, 2])
真实键活性分数: torch.Size([6, 2])
选中虚拟键数: 6
选中真实键数: 6
注意力层数: 原子级 3 层, 键级 3 层

虚拟键 Top-3 模板预测:
  键 (0,4): [(558, '0.0005'), (624, '0.0005'), (1940, '0.0005')]
  键 (0,3): [(558, '0.0005'), (624, '0.0005'), (1940, '0.0005')]
  键 (5,4): [(558, '0.0005'), (624, '0.0005'), (1940, '0.0005')]

真实键 Top-3 模板预测:
  键 (4,3): [(2128, '0.0003'), (1548, '0.0003'), (3304, '

## Step 7: 加载预训练模型进行推理

LocalTransform 提供了简洁的推理接口 `Synthesis.py`，封装了从 SMILES 到产物预测的完整流程。

### 推理流程

```text
反应物 SMILES
    │
    ├──► mol_to_bigraph()  → DGL 特征图
    ├──► get_adm()         → 原子距离矩阵
    └──► get_bonds()       → 虚拟/真实键列表
         │
         ▼
   model.forward() → (template_v, template_r, ...)
         │
         ├──► Softmax       → 模板概率
         ├──► combined_edit → 合并虚拟/真实键预测并排序
         │
         ▼
   Collector.collect()
         │
         ├──► 查表获取局部模板 SMARTS
         ├──► 在对应原子对上应用模板
         └──► 生成产物 SMILES
         │
         ▼
   Top-K 产物预测
```

### 模板应用示例

```text
预测: (type='r', bond=[2,3], template_id=42)
         │
         ├──► template_dicts['r'][42] = [
         │        '[C:1]-[O:2]>>[C:1]=O',
         │        'H_code', 'C_code', 'S_code', 'B'
         │    ]
         │
         ├──► 在原子 2-3 上应用“键断裂”模板
         │
         └──► Collector 执行模板变换 → 产物 SMILES
```

> **源码位置**: `Synthesis.py`，`class localtransform`、`predict_product()`

In [2]:
# ============================================================
# Step 7: 加载预训练模型进行推理
# ============================================================

# 切换到源码目录（Synthesis.py 使用相对路径加载模板和模型）
original_dir = os.getcwd()
os.chdir(LT_SOURCE)
print(f"工作目录: {os.getcwd()}")

# 加载模型权重
model_path = os.path.join(LT_SOURCE, 'models', 'LocalTransform_mix.pth')
print(f"模型权重: {model_path}")
print(f"模型大小: {os.path.getsize(model_path) / 1024 / 1024:.1f} MB")

# 使用 Synthesis.py 的接口
from Synthesis import localtransform

lt = localtransform(dataset='USPTO_480k', device='cpu')
print(f"\n✅ 模型加载完成")

工作目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/LocalTransform
模型权重: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/LocalTransform/models/LocalTransform_mix.pth
模型大小: 34.7 MB
loaded 4370 real templates
loaded 2535 virtual templates


Parameters of loaded LocalTransform:
{'attention_heads': 8, 'attention_layers': 3, 'edge_hidden_feats': 32, 'node_out_feats': 256, 'num_step_message_passing': 3, 'Template_rn': 4370, 'Template_vn': 2535, 'in_node_feats': 80, 'in_edge_feats': 13}

✅ 模型加载完成


In [11]:
# ============================================================
# 推理演示 1: 简单酯化反应
# ============================================================

# 反应物: 乙醇 + 乙酸 → 期望产物: 乙酸乙酯 + 水
reactant = 'CCO.CC(=O)O'
print(f"反应物 SMILES: {reactant}")
print(f"反应物: 乙醇 + 乙酸")
print()

# 预测产物 (Top-5)
results_df, results_dict = lt.predict_product(reactant, topk=5, verbose=1)

print(f"\n{'='*60}")
print("Top-5 预测结果:")
print('='*60)
for k in range(1, 6):
    key = f'Top-{k}'
    if reactant in results_dict and key in results_dict[reactant]:
        info = results_dict[reactant][key]
        product = info.get('product', '')
        score = info.get('score', 0)
        print(f"  {key}: {product} (score={score:.4f})")
    else:
        print(f"  {key}: (无预测)")

反应物 SMILES: CCO.CC(=O)O
反应物: 乙醇 + 乙酸

1th prediction: [A:1]=[A:2]-[A:3]>>[A:1]-[A:2]=[A:3] C [np.int64(6), np.int64(4)] 0.13543402
2th prediction: [A:1]=[A:2]-[A:3]>>[A:1]-[A:2]=[A:3] C [np.int64(5), np.int64(4)] 0.11520886
3th prediction: [A:1]=[A:2]-[A:3]>>[A:1]-[A:2]=[A:3] C [np.int64(5), np.int64(4)] 0.1129637
4th prediction: [A:1].[A:2]=[A:3]-[A:4]>>[A:1]-[A:2]-[A:3]=[A:4] C [np.int64(5), np.int64(4)] 0.10069128
5th prediction: [A:1].[A:2]-[A:3]>>[A:1]-[A:2]=[A:3] C [np.int64(1), np.int64(2)] 0.09458058
6th prediction: [A:1]=[A:2]-[A:3]>>[A:1]-[A:2]=[A:3] C [np.int64(6), np.int64(4)] 0.08538996
7th prediction: [A:1]=[A:2]-[A:3]>>[A:1]-[A:2]=[A:3] C [np.int64(4), np.int64(6)] 0.081703454
8th prediction: [A:1].[A:2]-[A:3]>>[A:1]-[A:2] A [np.int64(6), np.int64(1)] 0.07965857
9th prediction: [A:1]=[A:2]-[A:3]>>[A:1]-[A:2]=[A:3] C [np.int64(4), np.int64(6)] 0.06271966
10th prediction: [A:1].[A:3]-[A:2]=[A:4]>>[A:1]-[A:2]=[A:3] C [np.int64(4), np.int64(6)] 0.061402168
11th prediction: [

In [12]:
# ============================================================
# 推理演示 2: Wittig 反应
# ============================================================

reactant2 = 'O=Cc1ccccc1.C(=P(c1ccccc1)(c1ccccc1)c1ccccc1)C'
print(f"反应物: 苯甲醛 + Wittig 试剂")
print(f"SMILES: {reactant2}")
print()

results_df2, results_dict2 = lt.predict_product(reactant2, topk=5, verbose=1)

print(f"\n{'='*60}")
print("Top-5 预测结果:")
print('='*60)
for k in range(1, 6):
    key = f'Top-{k}'
    if reactant2 in results_dict2 and key in results_dict2[reactant2]:
        info = results_dict2[reactant2][key]
        product = info.get('product', '')
        score = info.get('score', 0)
        print(f"  {key}: {product} (score={score:.4f})")
    else:
        print(f"  {key}: (无预测)")

反应物: 苯甲醛 + Wittig 试剂
SMILES: O=Cc1ccccc1.C(=P(c1ccccc1)(c1ccccc1)c1ccccc1)C

1th prediction: [A:1]=[A:3].[A:2]=[A:4]>>[A:1]=[A:2] B [np.int64(1), np.int64(0)] 0.995793
2th prediction: [A:1]=[A:3].[A:2]=[A:4]>>[A:1]=[A:2] A [np.int64(8), np.int64(1)] 0.9936581
3th prediction: [A:1]=[A:3].[A:2]=[A:4]>>[A:1]=[A:2] B [np.int64(8), np.int64(9)] 0.9923105
4th prediction: [A:1]=[A:3].[A:2]=[A:4]>>[A:1]=[A:2] A [np.int64(1), np.int64(8)] 0.9832433
5th prediction: [A:1]=[A:3].[A:2]=[A:4]>>[A:1]=[A:2] A [np.int64(9), np.int64(0)] 0.027268073
6th prediction: [A:1]=[A:3].[A:2]=[A:4]>>[A:1]=[A:2] A [np.int64(0), np.int64(9)] 0.017378258
7th prediction: [A:1]=[A:3].[A:2]=[A:4]>>[A:1]=[A:2] B [np.int64(9), np.int64(8)] 0.00900696
8th prediction: [A:1]=[A:3].[A:2]=[A:4]>>[A:1]=[A:2] B [np.int64(0), np.int64(1)] 0.00636737
9th prediction: [A:1].[A:2]-[A:3]>>[A:1]-[A:2] A [np.int64(28), np.int64(2)] 0.0031725927
10th prediction: [A:1]=[A:3].[A:2]=[A:4]>>[A:1]-[A:2] A [np.int64(8), np.int64(1)] 0.0016236

In [13]:
# ============================================================
# 推理演示 3: 详细追踪推理过程
# ============================================================
from scripts.dataset import get_bonds, get_adm
from scripts.utils import pad_atom_distance_matrix, predict
from scripts.get_edit import combined_edit, get_bg_partition

# 手动执行推理过程，逐步观察
smiles = 'CCO.CC(=O)O'
mol = Chem.MolFromSmiles(smiles)

print(f"=== 逐步追踪推理过程 ===")
print(f"反应物: {smiles}\n")

# Step A: 构建图
graph = lt.graph_function(smiles)
adm = get_adm(mol)
v_bonds, r_bonds = get_bonds(smiles)
print(f"Step A - 图构建:")
print(f"  原子数: {mol.GetNumAtoms()}")
print(f"  虚拟键数: {len(v_bonds)}, 真实键数: {len(r_bonds)}")
print(f"  ADM shape: {adm.shape}, 范围: [{adm.min():.0f}, {adm.max():.0f}]")

# Step B: 模型前向传播
bg = dgl.batch([graph])
adms = pad_atom_distance_matrix([adm])
bonds_dicts_demo = {
    'virtual': [torch.from_numpy(v_bonds).long()],
    'real':    [torch.from_numpy(r_bonds).long()]
}

args_demo = {'device': torch.device('cpu')}
with torch.no_grad():
    pred_VT, pred_RT, _, _, pred_VI, pred_RI, attns = predict(args_demo, lt.model, bg, adms, bonds_dicts_demo)
    pred_VT_prob = nn.Softmax(dim=1)(pred_VT)
    pred_RT_prob = nn.Softmax(dim=1)(pred_RT)

print(f"\nStep B - 模型输出:")
print(f"  虚拟键模板概率: {pred_VT_prob.shape} (每行 {Template_vn+1} 类)")
print(f"  真实键模板概率: {pred_RT_prob.shape} (每行 {Template_rn+1} 类)")

# Step C: combined_edit 合并预测
v_sep, r_sep = get_bg_partition(bg, bonds_dicts_demo)
pred_vi = pred_VI[0].cpu()
pred_ri = pred_RI[0].cpu()
pred_v = pred_VT_prob[:v_sep[0]]
pred_r = pred_RT_prob[:r_sep[0]]

topk = 5
pred_types, pred_sites, pred_scores = combined_edit(
    v_bonds, r_bonds, pred_vi, pred_ri, pred_v, pred_r, topk * 10
)

print(f"\nStep C - combined_edit 合并排序:")
print(f"  共 {len(pred_types)} 个候选编辑")
for i in range(min(10, len(pred_types))):
    bond_type = pred_types[i]
    site = pred_sites[i]
    score = pred_scores[i]
    type_name = "虚拟键(形成)" if bond_type == 'v' else "真实键(断裂/变化)"
    print(f"  #{i+1}: {type_name} 原子对={site[0]}, 模板ID={site[1]}, score={score:.6f}")

# Step D: 模板查表
print(f"\nStep D - 模板查表 (前 3 个):")
for i in range(min(3, len(pred_types))):
    t = pred_types[i]
    template_id = pred_sites[i][1]
    if template_id in lt.template_dicts[t]:
        template_info = lt.template_dicts[t][template_id]
        template_smarts = template_info[0]
        action = template_info[-1]
        print(f"  #{i+1}: type={t}, template_id={template_id}")
        print(f"         SMARTS: {template_smarts}")
        print(f"         action: {action} ({'Addition' if action=='A' else 'Break' if action=='B' else 'Change' if action=='C' else 'Remove'})")

# 恢复目录
os.chdir(original_dir)
print(f"\n✅ 推理追踪完成")

=== 逐步追踪推理过程 ===
反应物: CCO.CC(=O)O

Step A - 图构建:
  原子数: 7
  虚拟键数: 32, 真实键数: 10
  ADM shape: (7, 7), 范围: [0, 7]

Step B - 模型输出:
  虚拟键模板概率: torch.Size([7, 2536]) (每行 2536 类)
  真实键模板概率: torch.Size([7, 4371]) (每行 4371 类)

Step C - combined_edit 合并排序:
  共 50 个候选编辑
  #1: 真实键(断裂/变化) 原子对=[np.int64(6), np.int64(4)], 模板ID=4319, score=0.135434
  #2: 真实键(断裂/变化) 原子对=[np.int64(5), np.int64(4)], 模板ID=4206, score=0.115209
  #3: 真实键(断裂/变化) 原子对=[np.int64(5), np.int64(4)], 模板ID=4319, score=0.112964
  #4: 真实键(断裂/变化) 原子对=[np.int64(5), np.int64(4)], 模板ID=4143, score=0.100691
  #5: 真实键(断裂/变化) 原子对=[np.int64(1), np.int64(2)], 模板ID=4220, score=0.094581
  #6: 真实键(断裂/变化) 原子对=[np.int64(6), np.int64(4)], 模板ID=4206, score=0.085390
  #7: 真实键(断裂/变化) 原子对=[np.int64(4), np.int64(6)], 模板ID=4319, score=0.081703
  #8: 虚拟键(形成) 原子对=[np.int64(6), np.int64(1)], 模板ID=2535, score=0.079659
  #9: 真实键(断裂/变化) 原子对=[np.int64(4), np.int64(6)], 模板ID=4206, score=0.062720
  #10: 真实键(断裂/变化) 原子对=[np.int64(4), np.int64(6)], 模板ID=4367, score=0

## 总结

### LocalTransform 模型架构回顾

| 组件 | 说明 | 参数 |
|------|------|------|
| `MPNNGNN` | DGL-Life 消息传递网络 | `node_out=256`, `edge_hidden=32`, `T=3` |
| `atom_att` (GRA) | 原子级全局反应性注意力 | 3 层 × 8 头，ADM 位置编码（8 类） |
| `pooling_v/r` | 虚拟键 / 真实键活性二分类 | `512→256→2` |
| `bondnet_v/r` | 虚拟键 / 真实键特征提取 | `512→256→256` |
| `bond_att` (GRA) | 键级全局反应性注意力 | 3 层 × 8 头，BDM 位置编码（2 类） |
| `output_v` | 虚拟键模板多分类 | `256→256→2536` |
| `output_r` | 真实键模板多分类 | `256→256→4371` |

### LocalTransform 的核心创新

1. **局部模板**：将全局反应模板分解为键级别的局部模板，每条键独立预测，突破了全局模板的组合爆炸问题。
2. **双轨预测**：虚拟键（键形成）和真实键（键变化 / 断裂）分开预测，各自有独立的模板库。
3. **图距离位置编码**：在注意力机制中使用 ADM / BDM 作为相对位置编码，使模型感知分子拓扑结构。
4. **反应性池化**：Top-K 筛选最可能的反应位点，避免对所有键做模板分类的计算开销。
5. **两阶段注意力**：先原子级聚合全局信息，再键级聚合活性键间的相互信息。

### 与其他方法的对比

| 方法 | 类型 | 粒度 | 模板数 | 特点 |
|------|------|------|--------|------|
| GLN | 全局模板 | 反应级 | ~11K | 模板覆盖完整反应 |
| LocalRetro | 局部模板（逆合成） | 原子级 | ~6.9K | 原子标注 + 编辑预测 |
| **LocalTransform** | **局部模板（正向）** | **键级** | **~6.9K** | **键级双轨预测 + 反应性池化** |
| MEGAN | 图编辑 | 动作级 | — | 自回归图编辑序列 |

### 完整教程回顾

- `1_环境配置.ipynb`：环境搭建、依赖验证、源码结构概览
- `2_数据处理.ipynb`：模板提取 → 反应标注 → 特征图构建 → 批量化
- `3_模型展示.ipynb`：MPNN → GRA → 反应性池化 → 完整模型 → 推理流程